# 🧠 Env

**Categoría:** 🧠 Deep Learning `(deep)`  
**Módulo:** `likelihood.models.deep.rl`  
**Clase:** `Env`  
**Fuente:** `likelihood/models/deep/rl.py:41`

> Wraps a Keras model to provide a reinforcement learning environment.


## 📖 Descripción

Wraps a Keras model to provide a reinforcement learning environment.

The environment samples actions from the model's predicted probability distribution and records
the resulting transitions for training.


## 🔧 Parámetros

| Parámetro | Tipo | Por defecto | Descripción |
| --- | --- | --- | --- |
| `model` | `Any` | `—` | — |
| `maxlen` | `int` | `100` | — |
| `name` | `str` | `'likenasium'` | — |


## 📋 Métodos disponibles

| Método | Origen | Descripción |
| --- | --- | --- |
| `step(state: np.ndarray, action: int, verbose: int = 0)` | propio | Perform an environment step with the given action. |
| `reset()` | propio | Reset the environment to initial state. |
| `get_transitions()` | propio | Get all stored transitions. |


## 🧩 Constructor

```python
from likelihood.models.deep.rl import Env

Env(model, maxlen: int = 100, name: str = 'likenasium')
```


## ⚡ Inicio rápido

```python
from likelihood.models.deep.rl import Env

model = Env(model=None)

# Ejemplo de uso
# model.get_transitions()

# Opcionales: maxlen=100, name='likenasium'
```


In [ ]:
from likelihood.models.deep.rl import Env

model = Env(model=None)

# Ejemplo de uso
# model.get_transitions()

# Opcionales: maxlen=100, name='likenasium'

## 🎛️ Ejemplo interactivo

Mueve los controles `maxlen`, `name` para reconfigurar el modelo.

> Requiere `ipywidgets` (`pip install ipywidgets`).


In [ ]:
# Ejemplo interactivo de Env
# Generado automáticamente a partir de la firma real de la clase.
from likelihood.models.deep.rl import Env
from ipywidgets import IntSlider, Text, interact

# 1) Datos de entrada requeridos: ajústalos a tu problema
model = None  # Any

# 2) Controles interactivos
widgets = {
    "maxlen": IntSlider(value=100, min=0, max=100, step=1, description="maxlen"),
    "name": Text(value="likenasium", description="name"),
}


def run_example(**overrides):
    """Construye Env con los valores elegidos."""
    values = {
        "maxlen": overrides["maxlen"],
        "name": overrides["name"],
        "model": model,
    }
    model = Env(**values)
    # Sugerencias: model.get_transitions()
    print("✅ Modelo construido:", type(model).__name__)
    return model


interact(run_example, **widgets);

## 🧾 Código fuente

<details>
<summary><code>likelihood/models/deep/rl.py</code></summary>

```python
class Env:
    """Wraps a Keras model to provide a reinforcement learning environment.

    The environment samples actions from the model's predicted probability distribution and records
    the resulting transitions for training.
    """

    def __init__(self, model: Any, maxlen: int = 100, name: str = "likenasium") -> None:
        """Initialize the environment with a model.

        Parameters
        ----------
        model : Any
            Model with `.predict()` method (e.g., Keras model).
        maxlen : int
            Maximum length of deque. By default it is set to `100`.
        name : str
            The name of the environment. By default it is set to `likenasium`.
        """
        self.model = model
        self.maxlen = maxlen
        self.transitions: deque = deque(
            maxlen=self.maxlen
        )  # Stores (state, action, reward, next_action, done)
        self.current_state: Optional[np.ndarray] = None
        self.current_step: int = 0
        self.done: bool = False

    def step(
        self, state: np.ndarray, action: int, verbose: int = 0
    ) -> Tuple[Any, int, float, Any, bool]:
        """Perform an environment step with the given action.

        Parameters
        ----------
        state : `np.ndarray`
            Current state to process (input to the model).
        action : `int`
            Expected action to process.

        Returns
        -------
        tuple : (current_state, action_pred, reward, next_action, done)
        """
        if self.done:
            return None, None, 0, None, True

        # Process action through model
        model_output = self.model.predict(state.reshape((1, -1)), verbose=verbose)
        action_pred = np.argmax(model_output, axis=1)[0]
        model_output[:, action_pred] = 0.0
        next_action = np.max(model_output, axis=1)[0]  # Second most probable action

        # Calculate reward (1 if correct prediction, 0 otherwise)
        reward = 1 if action_pred == action else 0

        # Update current state
        self.current_state = state
        self.current_step += 1

        # Add transition to history
        if self.current_step <= self.maxlen:
            self.transitions.append(
                (
                    self.current_state,  # Previous state
                    action_pred,  # Current action
                    reward,  # Reward
                    next_action,  # Next action
                    self.done,  # Done flag
                )
            )
        return self.current_state, action_pred, reward, next_action, self.done

    def reset(self) -> Optional[np.ndarray]:
        """Reset the environment to initial state."""
        self.current_state = None
        self.current_step = 0
        self.done = False
        self.transitions = deque(maxlen=self.maxlen)
        return self.current_state

    def get_transitions(self) -> deque:
        """Get all stored transitions."""
        return self.transitions
```

</details>
